In [ ]:
import pandas as pd

# Define the file path
file_path = "data/data-19403652.csv"

# 1. Select only the necessary columns to save memory
usecols = ["REPORTER", "PARTNER", "PRODUCT", "FLOW", "PERIOD", "INDICATORS", "INDICATOR_VALUE"]

# 2. Optimize data types to speed up loading and reduce RAM usage
dtypes = {
    "REPORTER": "category",
    "PARTNER": "category",
    "PRODUCT": "str",
    "FLOW": "int8",
    "PERIOD": "int32",
    "INDICATORS": "category",
    "INDICATOR_VALUE": "float64",
}


df_raw = pd.read_csv(file_path, usecols=usecols, dtype=dtypes)
print(f"Done! Total rows loaded: {len(df_raw):,}")

# 3. Filter for exports only (FLOW == 2) and drop the FLOW column
df_exp = df_raw[df_raw["FLOW"] == 2].drop(columns=["FLOW"]).copy()
print(f"Rows after filtering for exports: {len(df_exp):,}")

# 4. Pivot the dataset from long to wide format (separating VALUE_EUR and QUANTITY_KG)
df_wide = df_exp.pivot_table(
    index=["REPORTER", "PARTNER", "PRODUCT", "PERIOD"],
    columns="INDICATORS",
    values="INDICATOR_VALUE",
    observed=True
).reset_index()

# Remove the column index name to keep the dataframe clean
df_wide.columns.name = None

print(f"Rows after pivoting (final wide format): {len(df_wide):,}")
df_wide.head()

Loading raw data...
Done! Total rows loaded: 2,093,424
Rows after filtering for exports: 1,207,718
Rows after pivoting (final wide format): 603,859


,REPORTER,PARTNER,PRODUCT,PERIOD,QUANTITY_KG,VALUE_EUR
0,AT,AE,010121,201903,500.0,11628.0
1,AT,AE,481910,201901,6.0,58.0
2,AT,AE,481910,201902,0.0,28.0
3,AT,AE,481910,201903,0.0,10.0
4,AT,AE,481910,201904,3.0,96.0


In [13]:
import numpy as np

# 1. Extract time variables from the PERIOD column (e.g., 201908 -> Year: 2019, Month: 8)
df_wide["year"] = df_wide["PERIOD"] // 100
df_wide["month"] = df_wide["PERIOD"] % 100

# Create a proper datetime object for time series plotting
df_wide["date"] = pd.to_datetime(df_wide["PERIOD"].astype(str), format="%Y%m")
df_wide.head()

,REPORTER,PARTNER,PRODUCT,PERIOD,QUANTITY_KG,VALUE_EUR,year,month,date
0,AT,AE,010121,201903,500.0,11628.0,2019,3,2019-03-01
1,AT,AE,481910,201901,6.0,58.0,2019,1,2019-01-01
2,AT,AE,481910,201902,0.0,28.0,2019,2,2019-02-01
3,AT,AE,481910,201903,0.0,10.0,2019,3,2019-03-01
4,AT,AE,481910,201904,3.0,96.0,2019,4,2019-04-01
